**Core DataFrame & Column Skills (Basic)**

Q1: Create a DataFrame with schema enforcement

Problem: Create a DataFrame with user info ensuring proper types.

Constraints: Enforce schema, all types must match.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = SparkSession.builder.getOrCreate()

data = [
    (1, "Alice", 30),
    (2, "Bob", 25),
    (3, "Charlie", 35)
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

df = spark.createDataFrame(data, schema)
df.show()

In [0]:

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 30|
|  2|    Bob| 25|
|  3|Charlie| 35|
+---+-------+---+


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField,StringType,IntegerType

spark = SparkSession.builder.getOrCreate()

data = [
    (1, "Alice", 30),
    (2, "Bob", 25),
    (3, "Charlie", 35)
]

schema = StructType([
    StructField("id",IntegerType(),True),
    StructField("name",StringType(),True),
    StructField("age",StringType(),True),
])

df = spark.createDataFrame(data,schema)
df.show()

Q2: Add a literal column
Problem: Add a column country with value "USA" for all rows.

Expected Output:

In [0]:
+---+-------+---+-------+
| id|   name|age|country|
+---+-------+---+-------+
|  1|  Alice| 30|    USA|
|  2|    Bob| 25|    USA|
|  3|Charlie| 35|    USA|
+---+-------+---+-------+

In [0]:
from pyspark.sql.functions import col,lit
df2 = df.withColumn("country",lit("USA"))
df2.show()

Q3: Rename a column
Problem: Rename column name to full_name.

Expected Output:

In [0]:
+---+---------+---+
| id|full_name|age|
+---+---------+---+
|  1|    Alice| 30|
|  2|      Bob| 25|
|  3|  Charlie| 35|
+---+---------+---+


In [0]:
df3 = df.withColumnRenamed('name','full_name')
df3.show()

Q4: Use col and arithmetic operations
Problem: Create a column age_in_5_years = age + 5.

Expected Output:

In [0]:
+---+-------+---+------------+
| id|   name|age|age_in_5yrs|
+---+-------+---+------------+
|  1|  Alice| 30|          35|
|  2|    Bob| 25|          30|
|  3|Charlie| 35|          40|
+---+-------+---+------------+


In [0]:
df4 = df.withColumn('age_in_5_years',col('age') + 5)
df4.show()

Q5: Handle null values with fillna
Problem: Replace null values in age with 0.

In [0]:
data_with_null = [
    (1, "Alice", None),
    (2, "Bob", 25)
]

df_null = spark.createDataFrame(data_with_null, ["id", "name", "age"])
df_null.show()


In [0]:
df5 = df_null.fillna({'age':0})
df5.show()

In [0]:
+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice|  0|
|  2|  Bob| 25|
+---+-----+---+


Q6: Conditional column with when / otherwise
Problem: Add age_group column: "Adult" if age >= 30 else "Young".

Expected Output:

In [0]:
+---+-------+---+---------+
| id|   name|age|age_group|
+---+-------+---+---------+
|  1|  Alice| 30|    Adult|
|  2|    Bob| 25|     Young|
|  3|Charlie| 35|    Adult|
+---+-------+---+---------+


In [0]:
from pyspark.sql.functions import when
df6 = df.withColumn(
    'age_group',when(col('age') >= 30,'Adult').otherwise('Young')
)
df6.show()

Q7: Use expr for string concatenation
Problem: Create user_label = "User_" + name.

Expected Output:

In [0]:
+---+-------+---+----------+
| id|   name|age|user_label|
+---+-------+---+----------+
|  1|  Alice| 30|   User_Alice|
|  2|    Bob| 25|   User_Bob|
|  3|Charlie| 35|   User_Charlie|
+---+-------+---+----------+


In [0]:
from pyspark.sql.functions import expr

df7 = df.withColumn("user_label",expr("concat('User_',name)"))
df7.show()

Q8: Use coalesce to handle multiple null columns
Problem: coalesce(col1, col2, lit("Unknown")).

In [0]:
data_multi_null = [
    (1, None, None),
    (2, None, "ValueB"),
    (3, "ValueA", None)
]
df_multi_null = spark.createDataFrame(data_multi_null, ["id", "col1", "col2"])


In [0]:
+---+------+
| id|value |
+---+------+
|  1|Unknown|
|  2|ValueB |
|  3|ValueA |
+---+------+


In [0]:
from pyspark.sql.functions import coalesce, col, lit
df8 = df_multi_null.withColumn(
    "value",
    coalesce(col("col1"),col("col2"),lit("Unknown"))
)
df8.show()

Q9: Drop rows with null values
Problem: Remove all rows with any null values in age.

Expected Output:

In [0]:
+---+-----+---+
| id| name|age|
+---+-----+---+
|  2|  Bob| 25|
+---+-----+---+


In [0]:
df9 = df_null.dropna(subset = ["age"])
df9.show()

Q10: Select and alias multiple columns
Problem: Select id and name as username.

Expected Output:

In [0]:
+---+--------+
| id|username|
+---+--------+
|  1|   Alice|
|  2|     Bob|
|  3| Charlie|
+---+--------+


In [0]:
df10 = df.select(col("id"),col("name").alias("username"))
df10.show()

Q11: Chain multiple column operations
Problem: Add age_in_10 = age + 10, is_senior = age >= 60, greeting = "Hello " + name.

Expected Output:

In [0]:
+---+-------+---+----------+---------+---------+
| id|   name|age|age_in_10 |is_senior|greeting |
+---+-------+---+----------+---------+---------+
|  1|  Alice| 30|        40|    False|Hello Alice|
|  2|    Bob| 25|        35|    False|Hello Bob  |
|  3|Charlie| 35|        45|    False|Hello Charlie|
+---+-------+---+----------+---------+---------+


In [0]:
df11 = df.withColumn("age_in_10",col("age") + 10)\
         .withColumn("is_senior",col("age") >= 60) \
         .withColumn("greetings",expr("concat('Hello',' name')"))
df11.show()

Q12: Use expr for conditional math
Problem: bonus = 1000 if age > 30 else 500.

Expected Output:

In [0]:
+---+-------+---+-----+
| id|   name|age|bonus|
+---+-------+---+-----+
|  1|  Alice| 30|  500|
|  2|    Bob| 25|  500|
|  3|Charlie| 35| 1000|
+---+-------+---+-----+


In [0]:
df12 = df.withColumn("bonus", expr("CASE WHEN age > 30 THEN 1000 ELSE 500 END"))
df12.show()

Q13: Handle nested nulls with coalesce and when
Problem: Create final_value = col1 if not null else col2 if not null else "N/A".

Expected Output:

In [0]:
+---+------+------+
| id|col1  |col2  |final_value|
+---+------+------+
|  1| null | null | N/A       |
|  2| null | ValB | ValB      |
|  3| ValA | null | ValA      |
+---+------+------+


In [0]:
data_nested = [(1, None, None), (2, None, "ValB"), (3, "ValA", None)]
df_nested = spark.createDataFrame(data_nested, ["id", "col1", "col2"])

df13 = df_nested.withColumn(
    "final_value",
    coalesce(col("col1"), col("col2"), lit("N/A"))
)
df13.show()

Q14: Add multiple literal columns with different types
Problem: Add country="USA", score=100, is_active=True.

Expected Output:

In [0]:
+---+-----+---+-------+-----+---------+
| id| name|age|country|score|is_active|
+---+-----+---+-------+-----+---------+
|  1|Alice| 30|    USA|  100|     True|
|  2|  Bob| 25|    USA|  100|     True|
+---+-----+---+-------+-----+---------+


In [0]:
df14 = df_null.withColumn("country", lit("USA"))\
              .withColumn("score", lit(100))\
              .withColumn("is_active", lit(True))
df14.show()

Q15: Complex column expression with multiple operations
Problem: Create status = "Senior" if age>30 and score>90 else "Junior"

Expected Output:

In [0]:
+---+-----+---+-----+------+
| id| name|age|score|status|
+---+-----+---+-----+------+
|  1|Alice| 30| 100 |Junior|
|  2|  Bob| 25| 100 |Junior|
|  3|Charlie|35| 95  |Senior|
+---+-----+---+-----+------+


In [0]:
data_status = [(1, "Alice", 30, 100), (2, "Bob", 25, 100), (3, "Charlie", 35, 95)]
df_status = spark.createDataFrame(data_status, ["id", "name", "age", "score"])

df15 = df_status.withColumn(
    "status",
    when((col("age") > 30) & (col("score") > 90), "Senior").otherwise("Junior")
)
df15.show()
